In [ ]:
# Our standard imports for maths and basic methodology
import numpy as np
from sklearn.model_selection import train_test_split

# For user feedback
from tqdm import tqdm
import matplotlib.pyplot as plt

# Imports for pytorch
import torch
import torch.nn as nn

# import pathlib, urllib and zipfile
from pathlib import Path
import urllib.request
import zipfile

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# download and unzip datasets

# download zip file from websites(github)
def download(url, zip_name,zip_dir="data/zip_file" ):
    '''download zip file to zip_file'''
    zip_dir = Path(zip_dir)
    zip_dir.mkdir(parents=True, exist_ok=True)
    zip_path = zip_dir / zip_name

    if not zip_path.exists():
        urllib.request.urlretrieve(url, zip_path.as_posix())
        print('Save to:', zip_path)
    else:
        print('Zip already existd:', zip_path)

    return zip_path

def unzip(zip_path, folder_name, unzip_dir="data/unzip"):
    '''unzip zip files from zip foder under data directory to unzip
    folder under data directory. and each zip file will be saved in 
    each foder'''
    unzip_dir = Path(unzip_dir)
    unzip_dir.mkdir(parents=True, exist_ok=True)
    unzip_path = unzip_dir/folder_name
    unzip_path.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(Path(zip_path), 'r') as zf:
        zf.extractall(unzip_path)
    
    print('Unzip to:', unzip_path)
    return unzip_path



In [ ]:
# download and unzip 
# already downloaded, so in the last time I didn't run this cell, but it can work
url_en = "https://github.com/UniversalDependencies/UD_English-EWT/archive/refs/heads/master.zip"
url_zh = "https://github.com/UniversalDependencies/UD_Chinese-GSD/archive/refs/heads/master.zip"
url_ja = "https://github.com/UniversalDependencies/UD_Japanese-GSD/archive/refs/heads/master.zip"

zip_path_en = download(url_en, "en_ewt.zip")
unzip_path_en = unzip(zip_path_en, "en_ewt")

zip_path_zh = download(url_zh, "zh_gsd.zip")
unzip_path_zh = unzip(zip_path_zh, "zh_gsd")

zip_path_ja = download(url_ja, "ja_gsd.zip")
unzip_path_ja = unzip(zip_path_ja, "ja_gsd")

In [ ]:
unzip_path_en = Path("data/unzip/en_ewt")
unzip_path_zh = Path("data/unzip/zh_gsd")
unzip_path_ja = Path("data/unzip/ja_gsd")

print(unzip_path_en.exists(), unzip_path_zh.exists(), unzip_path_ja.exists())

In [ ]:
def find_conllu_files(folder):
    '''find conllu files in unzip folders'''
    folder = Path(folder)

    train_path = None
    dev_path = None
    test_path = None

    for f in folder.rglob("*.conllu"):
        if f.name.endswith("-ud-train.conllu"):
            train_path = f
        elif f.name.endswith("-ud-dev.conllu"):
            dev_path = f
        elif f.name.endswith("-ud-test.conllu"):
            test_path = f
    
    print("train:", train_path)
    print("dev:", dev_path)
    print("test:", test_path)

    return train_path, dev_path, test_path

en_train, en_dev, en_test = find_conllu_files(unzip_path_en)
ch_train, ch_dev, ch_test = find_conllu_files(unzip_path_zh)
ja_train, ja_dev, ja_test = find_conllu_files(unzip_path_ja)

In [ ]:
# parser

def parser(path):
    '''extrac tokens and tags from conllu files.
    X is a list containing lists of tokens of sentences
    y is a list containing lists of tags of sentences'''
    X = []
    y = []
    cur_x = []
    cur_y = []

    # open files
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            # skip comment line
            if line.startswith("#"):
                continue
            # sentence boundary
            if line == "":
                if len(cur_x) > 0:
                    X.append(cur_x)
                    y.append(cur_y)
                cur_x = []
                cur_y = []
                continue

            cleaned_line = line.split("\t")
            token = cleaned_line[1]
            tag = cleaned_line[3]
            cur_x.append(token)
            cur_y.append(tag)
        
        # last sentence boundary
        if len(cur_x) > 0:
            X.append(cur_x)
            y.append(cur_y)
    
    return X, y


In [ ]:
X_en_train, y_en_train = parser(en_train)
X_en_test, y_en_test = parser(en_test)

X_ch_train, y_ch_train = parser(ch_train)
X_ch_test, y_ch_test = parser(ch_test)

X_ja_train, y_ja_train = parser(ja_train)
X_ja_test, y_ja_test = parser(ja_test)

print("EN train/test:", len(X_en_train), len(X_en_test))
print("ZH train/test:", len(X_ch_train), len(X_ch_test))
print("JA train/test:", len(X_ja_train), len(X_ja_test))

In [ ]:
# building vocab

def build_token2idx(X_train):
    '''building vocab. input X_train is the list of lists of tokens of sentences,
    return a lookup table'''
    PAD = "<PAD>"
    UNK = "<UNK>"
    idx2token = {PAD: 0, UNK:1}

    for sent in X_train:
        for tok in sent:
            if tok not in idx2token:
                idx2token[tok] = len(idx2token)

    return idx2token

def build_tag2idx(y_train):
    '''same as function build_token2idx'''
    PAD = "<PAD>"
    tag2idx = {PAD: 0}
    
    for sent in y_train:
        for tag in sent:
            if tag not in tag2idx:
                tag2idx[tag] = len(tag2idx)
    return tag2idx


v = build_token2idx([["I","love","NLP"], ["I","love","Python"]])
print(v)
en_tag2idx = build_tag2idx(y_en_train)
print(len(en_tag2idx))


In [ ]:

def encode_tokens(X, token2idx):
    '''grab numbers related to tokens in token2idx, using these numbers rebuilding
    sentences as a sequence of numbers'''
    UNK = "<UNK>"
    unk_id = token2idx[UNK]

    X_ids = []
    for sent in X:
        sent_ids = []
        for tok in sent:
            if tok in token2idx:
                sent_ids.append(token2idx[tok])
            else:
                sent_ids.append(unk_id)
        X_ids.append(sent_ids)
    
    return X_ids

def encode_tags(y, tag2idx):
    '''same as encode_tokens, only objects are tags'''

    y_ids = []
    for sent in y:
        sent_ids = []
        for tag in sent:
            if tag in tag2idx:
                sent_ids.append(tag2idx[tag])
        y_ids.append(sent_ids)

    return y_ids

In [ ]:
# padding
def padding(X_ids, pad_id=0):
    '''pad all sequences in a batch, with same length'''
    lengths = [len(sent) for sent in X_ids] 
    max_len = max(lengths) 

    X_padded = []
    for sent in X_ids:
        padded = sent + [pad_id] * (max_len - len(sent))
        X_padded.append(padded)
    
    return X_padded, lengths

def mini_batch(X_ids, y_ids, batch_size, token_pad_id, tag_pad_id):
    ''' creat batches of token ids and tag ids, then pad them within each batch'''

    for i in range(0, len(X_ids), batch_size):
        X_batch = X_ids[i: i+batch_size]
        y_batch = y_ids[i: i+batch_size]

        X_padded, lengths = padding(X_batch, pad_id=token_pad_id)
        y_padded, _ = padding(y_batch, pad_id=tag_pad_id)

        X_tensor = torch.tensor(X_padded)
        y_tensor = torch.tensor(y_padded)
    
        yield X_tensor, y_tensor, lengths


In [ ]:
# custom dataset for pos tagging
from torch.utils.data import Dataset, DataLoader

class POSDataset(Dataset):
    '''stores encoded token and tag sequences'''
    def __init__(self, X_ids, y_ids):
        self.X_ids = X_ids
        self.y_ids = y_ids
    
    def __len__(self):
        '''retrun the number of sentences in the dataset'''
        return len(self.X_ids)
    
    def __getitem__(self, idx):
        '''return one input sentence and its tag sequence'''
        return self.X_ids[idx], self.y_ids[idx]

In [ ]:
# collate function for DataLoader
def collate_fn(batch, token_pad_id, tag_pad_id):
    X_batch = []
    y_batch = []

    # separate token and tag ids from the batch
    for x_ids, y_ids in batch:
        X_batch.append(x_ids)
        y_batch.append(y_ids)

    # pad token and tag sequences to the same length
    X_padded, lengths = padding(X_batch, pad_id=token_pad_id)
    y_padded, _ = padding(y_batch, pad_id=tag_pad_id)

    # convert padded sequences to tensors
    X_tensor = torch.tensor(X_padded)
    y_tensor = torch.tensor(y_padded)

    return X_tensor, y_tensor, lengths

In [ ]:
# build an RNN model with some extensions
class RNNTagger(nn.Module):

    def __init__(self, vocab_size, tag_size, pad_idx, emb_dim=128, hidden_dim=256, rnn_type="lstm", bidirectional=False, dropout=0.0):
        super().__init__()

        # map token ids to dense word embeddings
        self.embedding = nn.Embedding(
            vocab_size,
            emb_dim,
            padding_idx=pad_idx 
        )
        
        self.rnn_type = rnn_type
        self.bidirectional = bidirectional
        self.dropout = nn.Dropout(dropout) # to reduce overfitting

        if self.rnn_type == "lstm":
            self.rnn = nn.LSTM(
                input_size=emb_dim, # input size is embedding dimension
                hidden_size=hidden_dim, 
                batch_first=True, # input shape: (batch, seq_len, dim)
                bidirectional = bidirectional 
                )
        elif self.rnn_type == "gru":
            self.rnn = nn.GRU(
                input_size=emb_dim,
                hidden_size=hidden_dim,
                batch_first=True,
                bidirectional=bidirectional
            )
        
        # bidirectional RNN concatenates forwad and backward hidden states
        if self.bidirectional:
            fc_input_dim = hidden_dim * 2
        else:
            fc_input_dim = hidden_dim

        # map RNN outputs to POS tag scores
        self.fc = nn.Linear(fc_input_dim, tag_size)
        # Convert scores to log-probabilities for NLLLoss
        self.log_softmax = nn.LogSoftmax(dim=-1)
    
    def forward(self, X):
        X = self.embedding(X) # convert token ids into embedding vectors
        X = self.dropout(X) # apply dropout to embeddings

        X, _ = self.rnn(X) # run the recurrent layer over the sequence
        X = self.dropout(X) # apply dropout to outputs
        
        X = self.fc(X) # map hidden states to tag space
        X = self.log_softmax(X) # log-probability
        return X


In [ ]:
class POSTagger:
    def __init__(self, emb_dim=128, hidden_dim=256, lr=0.01, batch_size=32, num_epochs=5, rnn_type="lstm", bidirectional=False, dropout=0.0):
        self.emb_dim = emb_dim
        self.hidden_dim = hidden_dim
        self.lr = lr
        self.batch_size = batch_size
        self.num_epochs = num_epochs
        self.rnn_type = rnn_type
        self.bidirectional = bidirectional
        self.dropout = dropout

        self.token2idx = None
        self.tag2idx = None
        self.model = None
        self.idx2tag = None

    def fit(self, X_train, y_train):
        # build vocabularies from training data
        self.token2idx = build_token2idx(X_train)
        self.tag2idx = build_tag2idx(y_train)
        # reverse dictionary for predicted ids back to tag strings
        self.idx2tag = {idx: tag for tag, idx in self.tag2idx.items()}

        # convert tokens and tags to ids
        X_train_ids = encode_tokens(X_train, self.token2idx)
        y_train_ids = encode_tags(y_train, self.tag2idx)
        # wrap encoded data in dataset/dataloader
        train_dataset = POSDataset(X_train_ids, y_train_ids)
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            collate_fn=lambda batch: collate_fn(
                batch,
                token_pad_id=self.token2idx["<PAD>"],
                tag_pad_id=self.tag2idx["<PAD>"]
            )
        )

        # build the neural model
        self.model = RNNTagger(
            vocab_size=len(self.token2idx),
            tag_size=len(self.tag2idx),
            pad_idx=self.token2idx["<PAD>"],
            emb_dim=self.emb_dim,
            hidden_dim=self.hidden_dim,
            rnn_type=self.rnn_type,
            bidirectional=self.bidirectional,
            dropout=self.dropout
        ).to(device)

        # igonore pad position
        loss_fn = nn.NLLLoss(ignore_index=self.tag2idx["<PAD>"])
        # use Adam to optimise model parameters
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)

        # main training loop
        for epoch in range(self.num_epochs):
            total_correct = 0
            total_tokens = 0
            total_loss = 0
            num_batches = 0
            self.model.train()

            for X_batch, y_batch, lengths in train_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device
                                     )
                scores = self.model(X_batch) # forward pass
                
                # flatten batch and time dimentions for the loss function
                loss = loss_fn(
                    scores.view(-1, scores.shape[-1]),
                    y_batch.view(-1)
                )
                optimizer.zero_grad() # clear old gradients
                loss.backward() # backpropagation
                optimizer.step() # update model parameters

                total_loss += loss.item()
                num_batches += 1

                predictions = scores.argmax(dim=2) 
                mask = (y_batch != self.tag2idx["<PAD>"]) # ignore pad 

                # count correct predictions on real tokens
                correct = (predictions[mask] == y_batch[mask]).sum().item()
                total = mask.sum().item()

                total_correct += correct
                total_tokens += total
            
            average_loss = total_loss / num_batches
            average_accuracy = total_correct / total_tokens * 100
            
            print(f"epoch {epoch+1}: average_loss = {average_loss:.4f}, accuracy = {average_accuracy:.2f}%")
        
        return self
    
    def mini_batch_X(self, X_ids, batch_size, token_pad_id):
        # creat batches for prediction
        for i in range(0, len(X_ids), batch_size):
            X_batch = X_ids[i:i+batch_size]
            X_padded, lengths = padding(X_batch, pad_id=token_pad_id) # pad to some lengths
            X_tensor = torch.tensor(X_padded).to(device) # convert to tensor
            yield X_tensor, lengths

    def predict(self, X):
        X_ids = encode_tokens(X, self.token2idx) # encode raw tokens as ids

        self.model.eval() # switch model to evaluation mode
        all_predictions = []

        with torch.no_grad(): # no gradients needed during prediction
            for X_batch, lengths in self.mini_batch_X(
                X_ids,
                batch_size=self.batch_size,
                token_pad_id=self.token2idx["<PAD>"]
            ):
                scores = self.model(X_batch)
                predictions = scores.argmax(dim=2) # predicted tag ids

                # remove pad positions and decode ids back to strings
                for i, length in enumerate(lengths):
                    pred_ids = predictions[i][:length].tolist()
                    pred_tags = [self.idx2tag[idx] for idx in pred_ids]
                    all_predictions.append(pred_tags)
        
        return all_predictions


    def evaluate(self, X, y):
        X_ids = encode_tokens(X, self.token2idx) # encode input tokens
        y_ids = encode_tags(y, self.tag2idx) # encode gold tags

        self.model.eval() # evaluation mode
        total_correct = 0
        total_tokens = 0

        with torch.no_grad():
            for X_batch, y_batch, lengths in mini_batch(
                X_ids,
                y_ids,
                batch_size=self.batch_size,
                token_pad_id=self.token2idx["<PAD>"],
                tag_pad_id=self.tag2idx["<PAD>"]
            ):
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                scores = self.model(X_batch)
                predictions = scores.argmax(dim=2) # predicted tag ids

                mask = (y_batch != self.tag2idx["<PAD>"]) # ignore pads
                correct = (predictions[mask] == y_batch[mask]).sum().item()
                total = mask.sum().item()
                total_correct += correct
                total_tokens += total

        accuracy = total_correct / total_tokens * 100
        return accuracy
                

In [ ]:
# baseline: most frequency tag
from collections import Counter

def most_frequent_tag_baseline(y_train):
    """return the most frequent tag in the training set."""
    tag_counter = Counter()

    # count the frequency of each tag in the dataset
    for sent in y_train:
        for tag in sent:
            tag_counter[tag] += 1

    # select the tag with the highest frequency
    most_common_tag = tag_counter.most_common(1)[0][0]
    return most_common_tag


def evaluate_baseline(X, y, baseline_tag):
    """evaluate a baseline with token-level accuracy."""
    total_correct = 0
    total_tokens = 0

    # predict the true tag for every token in the test set
    for sent_x, sent_y in zip(X, y):
        for true_tag in sent_y:
            if true_tag == baseline_tag:
                total_correct += 1
            total_tokens += 1

    accuracy = total_correct / total_tokens * 100
    return accuracy

In [ ]:
# baseline tagger
en_baseline_tag = most_frequent_tag_baseline(y_en_train)
print("English baseline tag:", en_baseline_tag)

en_baseline_acc = evaluate_baseline(X_en_test, y_en_test, en_baseline_tag)
print(f"English baseline accuracy: {en_baseline_acc:.2f}%")

zh_baseline_tag = most_frequent_tag_baseline(y_ch_train)
print("Chinese baseline tag:", zh_baseline_tag)

zh_baseline_acc = evaluate_baseline(X_ch_test, y_ch_test, zh_baseline_tag)
print(f"Chinese baseline accuracy: {zh_baseline_acc:.2f}%")

ja_baseline_tag = most_frequent_tag_baseline(y_ja_train)
print("Japanese baseline tag:", ja_baseline_tag)

ja_baseline_acc = evaluate_baseline(X_ja_test, y_ja_test, ja_baseline_tag)
print(f"Japanese baseline accuracy: {ja_baseline_acc:.2f}%")

In [ ]:
# lstm tagger
tagger_lstm = POSTagger(
    emb_dim=128,
    hidden_dim=128,
    num_epochs=5,
    batch_size=32,
    rnn_type="lstm",
    bidirectional=False,
    dropout=0.0,
    lr=0.01
)

tagger_lstm.fit(X_en_train, y_en_train)
acc_lstm_en = tagger_lstm.evaluate(X_en_test, y_en_test)
print(f"LSTM English test accuracy: {acc_lstm_en:.2f}%")

tagger_lstm.fit(X_ch_train, y_ch_train)
acc_lstm_ch = tagger_lstm.evaluate(X_ch_test, y_ch_test)
print(f"LSTM Chinese test accuracy: {acc_lstm_ch:.2f}%")

tagger_lstm.fit(X_ja_train, y_ja_train)
acc_lstm_ja = tagger_lstm.evaluate(X_ja_test, y_ja_test)
print(f"LSTM Japanese test accuracy: {acc_lstm_ja:.2f}%")

In [ ]:
# gru tagger
tagger_gru = POSTagger(
    emb_dim=128,
    hidden_dim=128,
    num_epochs=5,
    batch_size=32,
    rnn_type="gru",
    bidirectional=False,
    dropout=0.0,
    lr=0.001
)

tagger_gru.fit(X_en_train, y_en_train)
acc_gru_en = tagger_gru.evaluate(X_en_test, y_en_test)
print(f"GRU English test accuracy: {acc_gru_en:.2f}%")

tagger_gru.fit(X_ch_train, y_ch_train)
acc_gru_ch = tagger_gru.evaluate(X_ch_test, y_ch_test)
print(f"GRU Chinese test accuracy: {acc_gru_ch:.2f}%")

tagger_gru.fit(X_ja_train, y_ja_train)
acc_gru_ja = tagger_gru.evaluate(X_ja_test, y_ja_test)
print(f"GRU Japanese test accuracy: {acc_gru_ja:.2f}%")

In [ ]:
# bilstm tagger
tagger_bilstm = POSTagger(
    emb_dim=128,
    hidden_dim=128,
    num_epochs=5,
    batch_size=32,
    rnn_type="lstm",
    bidirectional=True,
    dropout=0.0,
    lr=0.01
)

tagger_bilstm.fit(X_en_train, y_en_train)
acc_bilstm_en = tagger_bilstm.evaluate(X_en_test, y_en_test)
print(f"BiLSTM English test accuracy: {acc_bilstm_en:.2f}%")

tagger_bilstm.fit(X_ch_train, y_ch_train)
acc_bilstm_ch = tagger_bilstm.evaluate(X_ch_test, y_ch_test)
print(f"BiLSTM Chiese test accuracy: {acc_bilstm_ch:.2f}%")

tagger_bilstm.fit(X_ja_train, y_ja_train)
acc_bilstm_ja = tagger_bilstm.evaluate(X_ja_test, y_ja_test)
print(f"BiLSTM Japanese test accuracy: {acc_bilstm_ja:.2f}%")

In [ ]:
# bilstm with dropout
tagger_bilstm_drop = POSTagger(
    emb_dim=128,
    hidden_dim=128,
    num_epochs=5,
    batch_size=32,
    rnn_type="lstm",
    bidirectional=True,
    dropout=0.3,
    lr=0.01
)

tagger_bilstm_drop.fit(X_en_train, y_en_train)
acc_bilstm_drop_en = tagger_bilstm_drop.evaluate(X_en_test, y_en_test)
print(f"BiLSTM + Dropout English test accuracy: {acc_bilstm_drop_en:.2f}%")

tagger_bilstm_drop.fit(X_ch_train, y_ch_train)
acc_bilstm_drop_ch = tagger_bilstm_drop.evaluate(X_ch_test, y_ch_test)
print(f"BiLSTM + Dropout Chinese test accuracy: {acc_bilstm_drop_ch:.2f}%")

tagger_bilstm_drop.fit(X_ja_train, y_ja_train)
acc_bilstm_drop_ja = tagger_bilstm_drop.evaluate(X_ja_test, y_ja_test)
print(f"BiLSTM + Dropout Japanese test accuracy: {acc_bilstm_drop_ja:.2f}%")

In [ ]:
# Compute OOV rate

def compute_oov_rate(X_train, X_test):

    # Build vocabulary from training data
    vocab = set()
    for sent in X_train:
        for token in sent:
            vocab.add(token)

    total_tokens = 0
    oov_tokens = 0

    # Count tokens in test data
    for sent in X_test:
        for token in sent:
            total_tokens += 1
            if token not in vocab:
                oov_tokens += 1

    oov_rate = oov_tokens / total_tokens * 100

    return oov_rate, oov_tokens, total_tokens

In [ ]:
# ai used in this part!
print("\nFinal results")
print("-" * 60)
print(f"{'Model':<20} {'English':>10} {'Chinese':>10} {'Japanese':>10}")
print("-" * 60)
print(f"{'Baseline':<20} {en_baseline_acc:>10.2f} {zh_baseline_acc:>10.2f} {ja_baseline_acc:>10.2f}")
print(f"{'LSTM':<20} {acc_lstm_en:>10.2f} {acc_lstm_ch:>10.2f} {acc_lstm_ja:>10.2f}")
print(f"{'GRU':<20} {acc_gru_en:>10.2f} {acc_gru_ch:>10.2f} {acc_gru_ja:>10.2f}")
print(f"{'BiLSTM':<20} {acc_bilstm_en:>10.2f} {acc_bilstm_ch:>10.2f} {acc_bilstm_ja:>10.2f}")
print(f"{'BiLSTM+Dropout':<20} {acc_bilstm_drop_en:>10.2f} {acc_bilstm_drop_ch:>10.2f} {acc_bilstm_drop_ja:>10.2f}")
print("-" * 60)

In [ ]:
# English OOV
oov_en, oov_count_en, total_en = compute_oov_rate(X_en_train, X_en_test)
print(f"English OOV rate: {oov_en:.2f}% ({oov_count_en}/{total_en})")

# Chinese OOV
oov_ch, oov_count_ch, total_ch = compute_oov_rate(X_ch_train, X_ch_test)
print(f"Chinese OOV rate: {oov_ch:.2f}% ({oov_count_ch}/{total_ch})")

# Japanese OOV
oov_ja, oov_count_ja, total_ja = compute_oov_rate(X_ja_train, X_ja_test)
print(f"Japanese OOV rate: {oov_ja:.2f}% ({oov_count_ja}/{total_ja})")

# Analysis

In this experiment, we compared the performance of RNN models and baseline model in POS tagging task.The data we used is UD treebank, version2.17. 

The baseline model simply assigns the most frequenct tag in the training data to every token. As expected, this baseline performs poorly, achieving only 16.20% accuracy in English, 27.58% in Chinese, and 28.26% in Japanese. These results indicate that POS tagging cannot be solved by simple frequency-based approach and requires consideration of contexts.

All RNN models significantly outperform the baseline. The basic LSTM model alreay achieves strong performance, with test accuracies of 87.9% in English, 84.59% in Chinese and 91.30% in Japanese. This suggests that LSTM may have more ability at capturing sequential dependencies in language, which is much helpful in POS tagging task.

Comparing LSTM and GRU, the GRU model performs slightly worse in all three languages. This suggests that LSTM may provide a stronger ability to model long-range dependencies in this task.

Introducing bidirectional structure furthur improves performance. The BiLSTM model reaches 90.06% accuracy in English, 86.27% in Chinese and 95.09% in Japanese. This improvment is expected because POS tagging often depends on both preceding and following context in a sentence.

Finally, we add dropout to the BiLSTM model as a regularization technique. The results show a slight improvement in English and Japanese, but a slight decrease to 84.65% in Chinese. This indicates that regularization may help prevent overfitting, although it may vary on different datasets.

Overall, the neural models substantially outperform the baseline, and bidirectional structures achieve the best results. Differences across languages may partly be explained by OOV rates. Chinese has the highest OOV rate(12.46%), which may contribute to its lower overall accuracy compared to English and Japanese. Japanese has the lowest OOV rate(6.15%), which may help explain why it achieves the highest performance among the three languages. 
